# Lesson 1.1 - Programming Basics for AI Engineering

## Objectives
- Apply Python fundamentals (types, control flow, functions, modules, exceptions, I/O) to AI-oriented tasks.
- Build reusable, testable logic rather than one-off notebook snippets.
- Practice defensive programming patterns used in data/ML pipelines.

In [1]:
from dataclasses import dataclass
from pathlib import Path
import logging
from typing import Iterable

import pandas as pd
from sklearn.datasets import load_iris

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("lesson_1_1")

## Section A - Variables, Types, and Control Flow

Formal reminder:
- A variable is a name bound to an object.
- Control flow determines execution path (`if/elif/else`, loops, early return).

We will implement a small quality-gate pattern often used in production ETL.

In [2]:
@dataclass
class BatchQualityReport:
    total_rows: int
    valid_rows: int
    invalid_rows: int

    @property
    def invalid_ratio(self) -> float:
        if self.total_rows == 0:
            return 0.0
        return self.invalid_rows / self.total_rows


def quality_gate(values: Iterable[float], min_value: float = 0.0) -> BatchQualityReport:
    values = list(values)
    valid = [v for v in values if v >= min_value]
    invalid = len(values) - len(valid)
    report = BatchQualityReport(total_rows=len(values), valid_rows=len(valid), invalid_rows=invalid)
    return report


report = quality_gate([1.2, 0.4, -1.0, 3.3, 2.1], min_value=0.0)
print(report)
print("invalid_ratio:", round(report.invalid_ratio, 3))

BatchQualityReport(total_rows=5, valid_rows=4, invalid_rows=1)
invalid_ratio: 0.2


## Section B - Functions, Modules, and Error Handling

A robust function should:
- validate assumptions,
- raise precise exceptions,
- keep side-effects explicit.

In [3]:
class DataValidationError(ValueError):
    """Raised when input data violates contract constraints."""


def parse_positive_float(raw: str, field_name: str) -> float:
    try:
        value = float(raw)
    except ValueError as exc:
        raise DataValidationError(f"{field_name} must be numeric") from exc
    if value <= 0:
        raise DataValidationError(f"{field_name} must be > 0")
    return value


for raw in ["12.7", "abc", "-5"]:
    try:
        parsed = parse_positive_float(raw, "monthly_income")
        logger.info("Parsed monthly_income=%s", parsed)
    except DataValidationError as exc:
        logger.warning("Validation failed: %s", exc)

INFO | Parsed monthly_income=12.7


WARNING | Validation failed: monthly_income must be numeric


WARNING | Validation failed: monthly_income must be > 0


## Section C - Mini Project: File Processor for a Data Team

Scenario:
A team needs a daily species-level summary from incoming tabular data.

Pattern implemented:
1. Load data.
2. Validate required columns.
3. Compute derived feature.
4. Persist summary artifact.

In [4]:
REQUIRED_COLS = {"sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)", "target"}


def validate_columns(df: pd.DataFrame, required_cols: set[str]) -> None:
    missing = required_cols - set(df.columns)
    if missing:
        raise DataValidationError(f"Missing columns: {sorted(missing)}")


iris = load_iris(as_frame=True)
df = iris.frame.copy()
validate_columns(df, REQUIRED_COLS)

df["petal_area_proxy"] = df["petal length (cm)"] * df["petal width (cm)"]
species_lookup = dict(enumerate(iris.target_names))
df["species"] = df["target"].map(species_lookup)

summary = (
    df.groupby("species", as_index=False)
    .agg(
        mean_sepal_length=("sepal length (cm)", "mean"),
        mean_petal_area=("petal_area_proxy", "mean"),
        n=("species", "size"),
    )
    .sort_values("species")
)

out_dir = Path("../data")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "iris_summary.csv"
summary.to_csv(out_path, index=False)

print("Saved:", out_path)
display(summary)

Saved: ../data/iris_summary.csv


,species,mean_sepal_length,mean_petal_area,n
0,setosa,5.006,0.3656,50
1,versicolor,5.936,5.7204,50
2,virginica,6.588,11.2962,50


## Section D - Exercises with Solutions

### Exercise 1
Build a list of squared odd numbers from 1 to 15.

### Exercise 2
Refactor repeated logic into a reusable function.

In [5]:
# Exercise 1 solution
squared_odds = [x * x for x in range(1, 16) if x % 2 == 1]
print("squared_odds:", squared_odds)

# Exercise 2 solution
def summarize_numeric(values: list[float]) -> dict[str, float]:
    if not values:
        raise DataValidationError("values must not be empty")
    s = pd.Series(values, dtype="float64")
    return {
        "mean": float(s.mean()),
        "std": float(s.std(ddof=0)),
        "min": float(s.min()),
        "max": float(s.max()),
    }


print(summarize_numeric([2, 5, 8, 11]))

squared_odds: [1, 9, 25, 49, 81, 121, 169, 225]
{'mean': 6.5, 'std': 3.3541019662496847, 'min': 2.0, 'max': 11.0}


## Business Case Studies & Exceptions

- ETL validation gate: continue processing valid rows but fail job if invalid ratio exceeds threshold.
- Inference input validation: reject malformed payloads early with clear 4xx error contracts.
- Exception handling exception: for safety-critical systems, fail-fast may be preferable to degraded fallback.

## Interview Questions & Answers

1. Why does defensive programming matter in ML pipelines?
   Most failures occur at data boundaries, not in model equations.
2. When should you raise instead of fallback?
   Raise when invariant violations make output unsafe or misleading.
3. Why move logic from notebooks to modules?
   Better testability, reuse, and CI integration.